# 15.04 · Interpretability intro

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Read a transformer as a **residual stream** that layers read from and write to
- [ ] **Logit lens**: watch the prediction form layer by layer
- [ ] Find **induction heads** in GPT-2 small (the circuit behind in-context copying)
- [ ] **Activation patching**: locate which heads cause a behaviour (indirect object identification)
- [ ] Train a small **sparse autoencoder** and inspect its features; steer generation with an activation vector

## Resources
- [ARENA curriculum](https://www.arena.education/) (chapter 1: transformer interpretability) — free
- [TransformerLens](https://transformerlensorg.github.io/TransformerLens/) · [nnsight](https://nnsight.net/) · [SAELens](https://github.com/jbloomAus/SAELens) — free/OSS
- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) ·
  [In-context learning and induction heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) — free
- [Interpretability in the wild (IOI)](https://arxiv.org/abs/2211.00593) · [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features) ·
  [Neuronpedia](https://www.neuronpedia.org/) — free

We use plain PyTorch **hooks** on Hugging Face's GPT-2 (no extra library), so you see exactly what TransformerLens automates.
Mental model: `x = embed(tokens)`; then every block does `x = x + attn(x)` and `x = x + mlp(x)`; finally `logits = unembed(ln_f(x))`.
`x` is the **residual stream**: a shared 768-d workspace every component reads from and writes to.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer

from aieng import rag

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.set_grad_enabled(False)
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(device).eval()
L, H, D = model.config.n_layer, model.config.n_head, model.config.n_embd
print(f"GPT-2 small: {L} layers × {H} heads, d_model {D}")


def ids(text):
    return tok(text, return_tensors="pt").input_ids.to(device)

## 1. Logit lens
Apply the final layer norm + unembedding to the residual stream **after each layer**, as if the model stopped there.
You see when the right answer becomes the top guess.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
out = model(ids(prompt), output_hidden_states=True)
target = tok.encode(" Paris")[0]
probs = []
for layer, h in enumerate(out.hidden_states):  # 0 = embeddings, 1..12 = after each block
    x = h[0, -1] if layer == L else model.transformer.ln_f(h[0, -1])  # HF's last hidden state already has ln_f applied
    logits = model.lm_head(x)
    p = logits.softmax(-1)
    probs.append(p[target].item())
    print(f"layer {layer:2}: top {tok.decode(p.argmax())!r:>12}  p(' Paris') = {p[target]:.3f}")
plt.figure(figsize=(6, 2.5)); plt.plot(probs, "o-"); plt.xlabel("layer"); plt.ylabel("p(' Paris')"); plt.title("logit lens"); plt.show()

Early layers predict generic continuations (" the"); city names appear in the late-middle layers (wrong ones first: Rome, London) and
" Paris" wins only near the end. MLPs in middle layers have been found to store associations like "Eiffel Tower → Paris"; later layers
turn them into the output. GPT-2 small stays unsure (p(" Paris") peaks around 0.1–0.2): small models know facts weakly.

## 2. Induction heads
Show the model a random token sequence twice. On the second copy a trained model predicts the next token almost perfectly, by
looking back at what followed the same token last time. **Induction heads** implement this: at position *i* they attend to the
token *after* the previous occurrence of token *i*.

In [ ]:
torch.manual_seed(0)
T = 50
rand = torch.randint(1000, 20000, (8, T), device=device)
seq = torch.cat([torch.full((8, 1), tok.bos_token_id, device=device), rand, rand], dim=1)  # [BOS, A, A]
out = model(seq, output_attentions=True)
logp = out.logits.log_softmax(-1)[:, :-1].gather(-1, seq[:, 1:, None])[..., 0]
print(f"loss on first copy {-logp[:, :T].mean():.2f} vs second copy {-logp[:, T:].mean():.2f} (nats/token)")

# induction score: attention from position i (second copy) to position i - T + 1
scores = torch.zeros(L, H)
for layer, att in enumerate(out.attentions):  # att: (batch, heads, query, key)
    q = torch.arange(T + 1, 2 * T + 1, device=device)
    scores[layer] = att[:, :, q, q - T + 1].mean(dim=(0, 2)).cpu()
plt.figure(figsize=(6, 4)); plt.imshow(scores, cmap="viridis"); plt.colorbar(label="induction score")
plt.xlabel("head"); plt.ylabel("layer"); plt.title("induction heads in GPT-2 small"); plt.show()
top = torch.topk(scores.flatten(), 5)
print("top induction heads:", [f"L{i // H}H{i % H} ({v:.2f})" for v, i in zip(top.values.tolist(), top.indices.tolist())])

A few heads in layers 5–7 attend almost entirely to "the token after the last occurrence" — that's the copy mechanism behind much of
in-context learning. They need a **previous-token head** in an earlier layer to work (exercise 1).

## 3. Activation patching: what causes the answer?
Indirect object identification (IOI): *"When Mary and John went to the store, John gave a drink to"* → **Mary**.
Make a **corrupted** prompt where the answer flips (swap the second name) and measure the logit difference `logit(Mary) − logit(John)`.
Then run the corrupted prompt but **paste in** one activation from the clean run. If the answer comes back, that activation carries the
information. This is causal, unlike looking at attention patterns.

In [ ]:
clean = "When Mary and John went to the store, John gave a drink to"
corrupt = "When Mary and John went to the store, Mary gave a drink to"
c_ids, x_ids = ids(clean), ids(corrupt)
assert c_ids.shape == x_ids.shape
mary, john = tok.encode(" Mary")[0], tok.encode(" John")[0]


def logit_diff(logits):
    return (logits[0, -1, mary] - logits[0, -1, john]).item()


clean_out = model(c_ids, output_hidden_states=True)
clean_ld, corrupt_ld = logit_diff(clean_out.logits), logit_diff(model(x_ids).logits)
print(f"logit diff clean {clean_ld:.2f} | corrupted {corrupt_ld:.2f}")

# Head outputs are the input of each attention's output projection (c_proj), concatenated over heads.
clean_heads = {}
hooks = [model.transformer.h[l].attn.c_proj.register_forward_pre_hook(
    lambda m, a, l=l: clean_heads.__setitem__(l, a[0].clone())) for l in range(L)]
model(c_ids)
for h in hooks: h.remove()


def patch_head(layer, head):
    def hook(m, args):
        x = args[0].clone()
        sl = slice(head * D // H, (head + 1) * D // H)
        x[:, -1, sl] = clean_heads[layer][:, -1, sl]  # paste this head's clean output at the final position
        return (x,)
    handle = model.transformer.h[layer].attn.c_proj.register_forward_pre_hook(hook)
    ld = logit_diff(model(x_ids).logits)
    handle.remove()
    return (ld - corrupt_ld) / (clean_ld - corrupt_ld)  # 0 = no effect, 1 = fully restored


effect = torch.tensor([[patch_head(l, h) for h in range(H)] for l in range(L)])
plt.figure(figsize=(6, 4)); plt.imshow(effect, cmap="RdBu", vmin=-1, vmax=1); plt.colorbar(label="fraction of answer restored")
plt.xlabel("head"); plt.ylabel("layer"); plt.title("patching each head's output at the last token"); plt.show()
top = torch.topk(effect.flatten(), 5)
print("most important heads:", [f"L{i // H}H{i % H} ({v:.2f})" for v, i in zip(top.values.tolist(), top.indices.tolist())])

The top heads (L7H3, L7H9, L8H6, L8H10) are what the IOI paper calls **S-inhibition heads**: at the final position they write "John is the
repeated name, don't predict it". Later **name-mover heads** (L9H9, L9H6, L10H0) read that signal and copy the *other* name to the output —
which is why patching the name movers' output alone helps less here: in the corrupted run they're already following the wrong signal.
No single head restores more than ~30%: behaviours are spread over several components, which is why circuit work patches paths and
positions systematically (exercise 2).

## 4. Sparse autoencoders: finding features
Neurons are **polysemantic** (one neuron fires for many unrelated things) because models pack more features than dimensions
(**superposition**). A sparse autoencoder (SAE) re-expresses an activation as a sparse combination of many more learned directions,
which tend to be more interpretable. Train one on GPT-2's layer-6 residual stream over Wikipedia text:

In [ ]:
LAYER = 6
texts = rag.corpus()[:2500]
acts, toks = [], []
for t in texts:
    x = ids(t)[:, :128]
    h = model(x, output_hidden_states=True).hidden_states[LAYER][0, 1:]  # skip position 0 (huge-norm outlier)
    acts.append(h.float().cpu()); toks += x[0, 1:].tolist()
acts = torch.cat(acts)
print(f"{len(acts):,} activation vectors of size {acts.shape[1]}")
mean, scale = acts.mean(0), acts.norm(dim=1).mean()
A = ((acts - mean) / scale * np.sqrt(D)).to(device)  # centre and normalise


class SAE(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.enc, self.dec = nn.Linear(d, n), nn.Linear(n, d)
        self.dec.weight.data = nn.functional.normalize(self.dec.weight.data, dim=0)
        self.enc.weight.data = self.dec.weight.data.T.clone(); self.enc.bias.data.zero_()  # tied init: a standard trick for fewer dead features

    def forward(self, x):
        f = torch.relu(self.enc(x))
        return self.dec(f), f


N_FEAT, L1 = 8 * D, 3.0  # L1 strength sets sparsity: try 1 (denser, better reconstruction) and 5 (sparser)
sae = SAE(D, N_FEAT).to(device)
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)
with torch.enable_grad():
    for step in range(3000):
        x = A[torch.randint(len(A), (2048,), device=device)]
        rec, f = sae(x)
        loss = (rec - x).pow(2).sum(-1).mean() + L1 * (f * sae.dec.weight.norm(dim=0)).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0 or step == 2999:
            ev = 1 - (rec - x).pow(2).sum() / (x - x.mean(0)).pow(2).sum()
            print(f"step {step:4}: loss {loss.item():7.1f} | explained variance {ev:.2f} | L0 {(f > 0).float().sum(-1).mean():.0f} active features")

**Explained variance** (how much of the activation the SAE reconstructs) and **L0** (features active per token) trade off via the L1
coefficient. Now look at what makes individual features fire: the tokens (with a little context) where each feature is most active.

In [ ]:
with torch.no_grad():
    F_all = torch.cat([sae(A[i: i + 8192])[1].cpu() for i in range(0, len(A), 8192)])
freq = (F_all > 0).float().mean(0)
alive = ((freq > 1e-4) & (freq < 0.02)).nonzero().flatten()
print(f"{(freq == 0).sum()} dead features, {len(alive)} with activation frequency between 0.01% and 2%")


def context(i, w=6):
    return tok.decode(toks[max(0, i - w): i]) + "【" + tok.decode(toks[i]) + "】"


for feat in alive[torch.randperm(len(alive), generator=torch.Generator().manual_seed(0))[:8]].tolist():
    top = F_all[:, feat].topk(5).indices.tolist()
    print(f"feature {feat:5} (fires on {freq[feat]:.2%} of tokens):")
    for i in top:
        print("     ", repr(context(i))[1:-1][-60:])

With ~200k tokens and a few minutes of training, many features are already readable (a feature for years, for a specific word, for a
position in a list, …) and some are noise. Research SAEs train on billions of tokens with ~1M features; pre-trained SAEs for GPT-2, Gemma
(Gemma Scope) and Llama can be browsed on **Neuronpedia** and loaded with SAELens.

## 5. Steering: write into the residual stream
If directions in the residual stream mean things, adding a direction should change behaviour. **Activation addition**: take the difference
between activations for two contrasting prompts and add it (scaled) during generation.

In [ ]:
def resid(text, layer=LAYER):
    return model(ids(text), output_hidden_states=True).hidden_states[layer][0, -1]


steer = resid(" I love this. It is wonderful and happy") - resid(" I hate this. It is terrible and sad")


def generate(prompt, coef=0.0, n=30, seed=0):
    def hook(m, args, out):  # block output is a tensor (or a tuple whose first element is the hidden state)
        return (out[0] + coef * steer,) + tuple(out[1:]) if isinstance(out, tuple) else out + coef * steer
    handle = model.transformer.h[LAYER - 1].register_forward_hook(hook)
    torch.manual_seed(seed)
    out = model.generate(ids(prompt), max_new_tokens=n, do_sample=True, top_p=0.9, temperature=0.8, pad_token_id=tok.eos_token_id)
    handle.remove()
    return tok.decode(out[0][ids(prompt).shape[1]:])


for coef in (-1, 0, 1):
    print(f"coef {coef:+}: I went to the restaurant yesterday and{generate('I went to the restaurant yesterday and', coef)!r}")

The steering vector's norm is ~10% of the residual stream's; much larger coefficients (try ±4) break the model into gibberish — it pushes
activations off the distribution the later layers were trained on.

Steering is used for research (does this direction *cause* the behaviour?) and as a lightweight control (refusal directions, persona
vectors). It is also a warning: anyone with the weights can edit behaviour this way, so open-weight safety can't rely on refusals alone.

## Exercises
1. Find an induction head in GPT-2 small (done above) — then find its partner **previous-token head**: score each head by attention from
   position *i* to *i − 1*. Which layer is it in, and is it earlier than the induction heads?
2. Patch the **residual stream** at every (layer, position) for the IOI prompt and plot the heatmap. Where does the name information move
   from the "John" token to the final position?
3. Increase the SAE to 16× features or lower L1; plot explained variance vs L0. Pick 10 features and write a one-line label for each from
   their top activations. How many are clearly interpretable?
4. Reproduce §1 with **tuned lens** or on a bigger model (`Qwen/Qwen3-0.6B`): at which relative depth do facts appear?

In [ ]:
# your code here